LLM - huggingface LLM (default : gpt-3.5-turbo)
https://docs.llamaindex.ai/en/stable/module_guides/models/llms/usage_custom/

In [ ]:
from llama_index.core import PromptTemplate


# Transform a string into input zephyr-specific input
def completion_to_prompt(completion):
    return f"<|system|>\n</s>\n<|user|>\n{completion}</s>\n<|assistant|>\n"


# Transform a list of chat messages into zephyr-specific input
def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|system|>\n{message.content}</s>\n"
        elif message.role == "user":
            prompt += f"<|user|>\n{message.content}</s>\n"
        elif message.role == "assistant":
            prompt += f"<|assistant|>\n{message.content}</s>\n"

    # ensure we start with a system prompt, insert blank if needed
    if not prompt.startswith("<|system|>\n"):
        prompt = "<|system|>\n</s>\n" + prompt

    # add final assistant prompt
    prompt = prompt + "<|assistant|>\n"

    return prompt

In [ ]:
import torch
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings

Settings.llm = HuggingFaceLLM(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    tokenizer_name="HuggingFaceH4/zephyr-7b-beta",
    context_window=3900,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0, "top_k": 50, "top_p": 0.95},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto",
)

EMBEDDING MODEL - BAAI (default : text-embedding-ada-002)
https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings/

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

VECTOR STORE 

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Load documents and build index
documents = SimpleDirectoryReader(
    "/home/jjh_test/llama_index/data"
).load_data()
index = VectorStoreIndex.from_documents(documents)

INDEX SETUP

In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [ ]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex.from_documents(documents)

QUERY ENGINE

In [ ]:
query_engine = vector_index.as_query_engine(response_mode="compact")

EVALUATION

In [ ]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [ ]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [ ]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [ ]:
responses_str = []
responses = []
count=0

In [ ]:
for question in questions:
    count+=1
    query = f"Do not repeat the question.{question}. Start the answer with 'True' or 'False'."
    response = query_engine.query(query)
    responses.append(response)
    
    response_str=str(response)
    print(count, response)
    if "True" in response_str:
        response_str="True"
    elif "False" in response_str:
        response_str="False"
    else:
        print("error")
    responses_str.append(response_str)

In [ ]:
correct_count=0
number=0

for response, answer, response_str, question in zip(responses, answers, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

In [ ]:
accuracy = (correct_count / len(questions)) * 100

In [ ]:
print(f"model_name: ", Settings.llm.model)
print(f"Embedding Model Name: {Settings.embed_model.model_name}")
print(f"VectorStore model: {index.vector_store.model_name}")
print()

print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")